In [ ]:
import pandas as pd

In [ ]:
df_plot = pd.read_csv("/content/drive/MyDrive/data_movie/data_movie/plot_details.csv")

In [ ]:
df_plot.head()

,Release Year,Title,Plot
0,1901,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,The earliest known adaptation of the classic f...


In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opente

In [ ]:
from sentence_transformers import SentenceTransformer, util
from sentence_transformers import SentenceTransformer

### Initializing sentence embedding model
model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name, device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import chromadb

chroma_db_path = '/content/drive/MyDrive/data_movie/data_movie/plotDB_2'

client = chromadb.PersistentClient(path=chroma_db_path)

In [ ]:
import re

def clean_text(desc):
  modified_text = desc.replace("<ul>", "").replace("</ul>", "").replace("<li>", "").replace("</li>", ",").replace("<br>", ".")
  final = re.sub(r'<[^>]*>', '', modified_text).replace(". .",".")
  return final

In [ ]:
df_plot["Plot"] = df_plot["Plot"].apply(clean_text)

In [ ]:
def truncate_text(text, max_words=200):
    return " ".join(text.split()[:max_words])

df_plot['short_plot'] = df_plot['Plot'].apply(truncate_text)

In [ ]:
df_plot["metadata"] = df_plot.apply(lambda x: {
    "Title": x["Title"],
    "Plot": x["Plot"],
    "Release_Year": x["Release Year"],
    "short_plot": x["short_plot"]
}, axis=1)

In [ ]:

def get_search_text(metadata):
  text = ""
  ### Adding product description
  text += metadata["short_plot"] + " "

  if len(text) == 0:
    text = "No available description"

  return text

In [ ]:
df_plot["text_search"] = df_plot["metadata"].apply(get_search_text)

In [ ]:
df_plot.head()

,Release Year,Title,Plot,short_plot,metadata,text_search
0,1901,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr...","A bartender is working at a saloon, serving dr...","{'Title': 'Kansas Saloon Smashers', 'Plot': 'A...","A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov...","The moon, painted with a smiling face hangs ov...","{'Title': 'Love by the Light of the Moon', 'Pl...","The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,"The film, just over a minute long, is composed...","The film, just over a minute long, is composed...","{'Title': 'The Martyred Presidents', 'Plot': '...","The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...,Lasting just 61 seconds and consisting of two ...,"{'Title': 'Terrible Teddy, the Grizzly King', ...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,The earliest known adaptation of the classic f...,The earliest known adaptation of the classic f...,"{'Title': 'Jack and the Beanstalk', 'Plot': 'T...",The earliest known adaptation of the classic f...


In [ ]:
len(df_plot)

34886

In [ ]:
df_plot= df_plot[df_plot["Release Year"]> 1950]

In [ ]:
len(df_plot)

28360

In [ ]:
def generate_embeddings(texts):
    embeddings = embedder.encode(texts, convert_to_tensor=False)
    return embeddings

In [ ]:
embeddings = embedder.encode(
    df_plot['short_plot'].tolist(),
    batch_size=128,
    show_progress_bar=True
)

Batches:   0%|          | 0/222 [00:00<?, ?it/s]

In [ ]:
df_plot["embeddings"] = embeddings.tolist()

In [ ]:
df_plot.to_csv("drive/MyDrive/data_movie/data_movie/movie_plot_mod.csv", index=False)

In [ ]:
collection = client.get_or_create_collection(name='Movie_plot')

In [ ]:
ids = [str(i) for i in range(len(df_plot))]

In [ ]:
max_batch_size = 5000  # safe margin

for i in range(0, len(df_plot), max_batch_size):
    batch = df_plot.iloc[i:i+max_batch_size]

    collection.add(
        embeddings=batch['embeddings'].tolist(),
        documents=batch['text_search'].tolist(),
        metadatas=batch['metadata'].tolist(),
        ids=ids[i:i+max_batch_size]
    )

In [ ]:
cache_collection_name = 'Cache'

In [ ]:
cache_collection = client.get_or_create_collection(name=cache_collection_name)

In [ ]:
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def get_movie_plot(movie_name):
    base_url = "https://en.wikipedia.org/w/api.php"

    # Step 1: Search correct page
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": movie_name + " film",
        "format": "json"
    }

    res = requests.get(base_url, params=search_params, headers=HEADERS)
    data = res.json()

    results = data.get("query", {}).get("search", [])
    if not results:
        return "Movie not found"

    page_title = results[0]["title"]

    # Step 2: Get sections
    section_params = {
        "action": "parse",
        "page": page_title,
        "prop": "sections",
        "format": "json"
    }

    res = requests.get(base_url, params=section_params, headers=HEADERS)
    data = res.json()

    sections = data.get("parse", {}).get("sections", [])

    # Step 3: Find plot section index
    plot_index = None
    for sec in sections:
        title = sec["line"].lower()
        if any(k in title for k in ["plot", "synopsis", "premise"]):
            plot_index = sec["index"]
            break

    if not plot_index:
        return "Plot section not found"

    # Step 4: Fetch that section content
    content_params = {
        "action": "parse",
        "page": page_title,
        "prop": "text",
        "section": plot_index,
        "format": "json"
    }

    res = requests.get(base_url, params=content_params, headers=HEADERS)
    data = res.json()

    html = data["parse"]["text"]["*"]

    # Step 5: Clean HTML → text
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")

    paragraphs = [p.get_text() for p in soup.find_all("p")]

    return "\n".join(paragraphs).strip()


# Test
print(get_movie_plot("Inception"))
print(get_movie_plot("Avatar"))
print(get_movie_plot("The Dark Knight"))

Dom Cobb and Arthur are "extractors" who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets' subconscious and extract information. Their latest target, Saito, is impressed with Cobb's ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of implanting an idea into a person's subconscious; performing "inception" on Robert Fischer, the son of Saito's competitor Maurice Fischer, with the idea to dissolve his father's company. In return, Saito promises to clear Cobb's criminal status, allowing him to return home to his children.

Cobb accepts the offer and assembles his team: a forger named Eames, a chemist named Yusuf, and a college student named Ariadne. Ariadne is tasked with designing the dream's architecture, something Cobb himself cannot do for fear of being sabotaged by his mind's projection of his late wife, Mal. Maurice Fischer dies, and the team sedates Robert Fischer into a th

In [ ]:
movie_target = get_movie_plot("Inception")

In [ ]:
movie_target

'Dom Cobb and Arthur are "extractors" who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets\' subconscious and extract information. Their latest target, Saito, is impressed with Cobb\'s ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of implanting an idea into a person\'s subconscious; performing "inception" on Robert Fischer, the son of Saito\'s competitor Maurice Fischer, with the idea to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s criminal status, allowing him to return home to his children.\n\nCobb accepts the offer and assembles his team: a forger named Eames, a chemist named Yusuf, and a college student named Ariadne. Ariadne is tasked with designing the dream\'s architecture, something Cobb himself cannot do for fear of being sabotaged by his mind\'s projection of his late wife, Mal. Maurice Fischer dies, and the team sedates Robert Fische

In [ ]:
cleaned_text = clean_text(movie_target)

In [ ]:
cleaned_text

'Dom Cobb and Arthur are "extractors" who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets\' subconscious and extract information. Their latest target, Saito, is impressed with Cobb\'s ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of implanting an idea into a person\'s subconscious; performing "inception" on Robert Fischer, the son of Saito\'s competitor Maurice Fischer, with the idea to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s criminal status, allowing him to return home to his children.\n\nCobb accepts the offer and assembles his team: a forger named Eames, a chemist named Yusuf, and a college student named Ariadne. Ariadne is tasked with designing the dream\'s architecture, something Cobb himself cannot do for fear of being sabotaged by his mind\'s projection of his late wife, Mal. Maurice Fischer dies, and the team sedates Robert Fische

In [ ]:
short_plot = truncate_text(cleaned_text)

In [ ]:
short_plot

'Dom Cobb and Arthur are "extractors" who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets\' subconscious and extract information. Their latest target, Saito, is impressed with Cobb\'s ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of implanting an idea into a person\'s subconscious; performing "inception" on Robert Fischer, the son of Saito\'s competitor Maurice Fischer, with the idea to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s criminal status, allowing him to return home to his children. Cobb accepts the offer and assembles his team: a forger named Eames, a chemist named Yusuf, and a college student named Ariadne. Ariadne is tasked with designing the dream\'s architecture, something Cobb himself cannot do for fear of being sabotaged by his mind\'s projection of his late wife, Mal. Maurice Fischer dies, and the team sedates Robert Fischer i

In [ ]:
query = f"""

Find movie plots similar to:

{short_plot}

"""

In [ ]:
### Getting first 10 matches

results = collection.query(
query_texts=query,
n_results=10
)
results.items()

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.8MiB/s]


dict_items([('ids', [['10570', '14486', '21094', '15871', '9932', '11998', '5263', '11643', '24823', '15830']]), ('embeddings', None), ('documents', [['Dominick "Dom" Cobb and Arthur are "extractors", who perform corporate espionage using an experimental military technology to infiltrate the subconscious of their targets and extract valuable information through a shared dream world. Their latest target, Japanese businessman Saito, reveals that he arranged their mission himself to test Cobb for a seemingly impossible job: planting an idea in a person\'s subconscious, or "inception". To break up the energy conglomerate of ailing competitor Maurice Fischer, Saito wants Cobb to convince Fischer\'s son and heir, Robert, to dissolve his father\'s company. In return, Saito promises to use his influence to clear Cobb of a murder charge, allowing Cobb to return home to his children. Cobb accepts the offer and assembles his team: Eames, a conman and identity forger; Yusuf, a chemist who concocts

In [ ]:
results

{'ids': [['10570',
   '14486',
   '21094',
   '15871',
   '9932',
   '11998',
   '5263',
   '11643',
   '24823',
   '15830']],
 'embeddings': None,
 'documents': [['Dominick "Dom" Cobb and Arthur are "extractors", who perform corporate espionage using an experimental military technology to infiltrate the subconscious of their targets and extract valuable information through a shared dream world. Their latest target, Japanese businessman Saito, reveals that he arranged their mission himself to test Cobb for a seemingly impossible job: planting an idea in a person\'s subconscious, or "inception". To break up the energy conglomerate of ailing competitor Maurice Fischer, Saito wants Cobb to convince Fischer\'s son and heir, Robert, to dissolve his father\'s company. In return, Saito promises to use his influence to clear Cobb of a murder charge, allowing Cobb to return home to his children. Cobb accepts the offer and assembles his team: Eames, a conman and identity forger; Yusuf, a chemist

In [ ]:
cache_results = cache_collection.query(
    query_texts = [query],
    n_results = 2
)


In [ ]:
# Implementing Cache in Semantic Search

# Set a threshold for cache search
threshold = 0.2

ids = []
documents = []
distances = []
metadatas = []
results_df = pd.DataFrame()

# Check if the distance is greater than the threshold, if so, return results from the main collection
if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
    # Query the collection against the user query and return the results
    results = collection.query(
        query_texts=query,
        n_results=5
    )

    # Store the query in cache_collection as a document with respect to ChromaDB for future reference
    # Store retrieved text, ids, distances, and metadatas in cache_collection as metadatas, so they can be fetched easily if a query indeed matches to a query in cache
    Keys = []
    Values = []

    for key, val in results.items():
        if val is None:
            continue
        for i in range(len(val[0])):  # Iterate over the actual length of val
            Keys.append(str(key) + str(i))
            if len(val[0]) > i:  # Check if the current index exists in val
                Values.append(str(val[0][i]))

    cache_collection.add(
        documents=[query],
        ids=[query],
        metadatas=dict(zip(Keys, Values))
    )

    # Print message indicating the results are found in the main collection
    print("Not found in cache. Found in the main collection.")

    # Construct a DataFrame from the query results
    result_dict = {'Metadatas': results['metadatas'][0], 'Documents': results['documents'][0], 'Distances': results['distances'][0], "IDs": results["ids"][0]}
    results_df = pd.DataFrame.from_dict(result_dict)


# If the distance is less than the threshold, return results from the cache
elif cache_results['distances'][0][0] <= threshold and cache_results['ids']:
    cache_result_dict = cache_results['metadatas'][0][0]

    # Loop through each inner list and then through the dictionary
    for key, value in cache_result_dict.items():
        if 'ids' in key:
            ids.append(value)
        elif 'documents' in key:
            documents.append(value)
        elif 'distances' in key:
            distances.append(value)
        elif 'metadatas' in key:
            metadatas.append(value)

    # Print message indicating the results are found in the cache
    print("Found in cache!")

    # Create a DataFrame from the cached results
    results_df = pd.DataFrame({
        'IDs': ids,
        'Documents': documents,
        'Distances': distances,
        'Metadatas': metadatas
    })
else:
    # Print message indicating no valid results found in cache
    print("No valid results found in cache!")



Not found in cache. Found in the main collection.


In [ ]:
results_df.head()

,Metadatas,Documents,Distances,IDs
0,"{'Release_Year': 2010, 'short_plot': 'Dominick...","Dominick ""Dom"" Cobb and Arthur are ""extractors...",0.401830,10570
1,"{'Release_Year': 1998, 'Title': 'Lucia', 'Plot...","The plot is non-linear, and the end scene of t...",0.885444,14486
2,"{'Plot': 'The plot is non-linear, and the end ...","The plot is non-linear, and the end scene of t...",0.885444,21094
3,"{'short_plot': 'The film starts with a man, Ry...","The film starts with a man, Ryjkin, trying to ...",0.905679,15871
4,"{'Release_Year': 2007, 'Title': 'The Good Nigh...",The movie follows a man's search for perfectio...,0.986720,9932


In [ ]:
from sentence_transformers import CrossEncoder, util

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
### Generating cross_encoder scores query

cross_inputs = [[query, response] for response in results_df['Documents']]
cross_rerank_scores = cross_encoder.predict(cross_inputs)

results_df['Reranked_scores'] = cross_rerank_scores

In [ ]:

semantic = results_df.sort_values(by='Distances')
semantic[:3]

,Metadatas,Documents,Distances,IDs,Reranked_scores
0,"{'Release_Year': 2010, 'short_plot': 'Dominick...","Dominick ""Dom"" Cobb and Arthur are ""extractors...",0.401830,10570,3.257682
1,"{'Release_Year': 1998, 'Title': 'Lucia', 'Plot...","The plot is non-linear, and the end scene of t...",0.885444,14486,-7.524955
2,"{'Plot': 'The plot is non-linear, and the end ...","The plot is non-linear, and the end scene of t...",0.885444,21094,-7.524955


In [ ]:
## Rank based

rank = results_df.sort_values(by='Reranked_scores')
rank[:3]

,Metadatas,Documents,Distances,IDs,Reranked_scores
4,"{'Release_Year': 2007, 'Title': 'The Good Nigh...",The movie follows a man's search for perfectio...,0.986720,9932,-7.952139
1,"{'Release_Year': 1998, 'Title': 'Lucia', 'Plot...","The plot is non-linear, and the end scene of t...",0.885444,14486,-7.524955
2,"{'Plot': 'The plot is non-linear, and the end ...","The plot is non-linear, and the end scene of t...",0.885444,21094,-7.524955


In [ ]:
final = rank[["Documents", "Metadatas", "IDs"]]

In [ ]:
final

,Documents,Metadatas,IDs
4,The movie follows a man's search for perfectio...,"{'Release_Year': 2007, 'Title': 'The Good Nigh...",9932
1,"The plot is non-linear, and the end scene of t...","{'Release_Year': 1998, 'Title': 'Lucia', 'Plot...",14486
2,"The plot is non-linear, and the end scene of t...","{'Plot': 'The plot is non-linear, and the end ...",21094
3,"The film starts with a man, Ryjkin, trying to ...","{'short_plot': 'The film starts with a man, Ry...",15871
0,"Dominick ""Dom"" Cobb and Arthur are ""extractors...","{'Release_Year': 2010, 'short_plot': 'Dominick...",10570
